# Lesson 3

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import anthropic
client = anthropic.Anthropic()

In [ ]:
resp = client.messages.create(
    model="claude-sonnet-5-5", #haiku #sonnet #Opus #Fable
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": "Introduce yourself in one sentence."
        }
    ]
)

In [ ]:
resp

In [ ]:
import json

print(json.dumps(resp.model_dump(), indent=2))

In [ ]:
response = resp.content[0].text

In [ ]:
print(response)

In [ ]:
# Try to call Claude
try:
    resp = client.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=200,
        messages=[
            {
                "role": "user",
                "content": "Introduce yourself in one sentence."
            }
        ]
    )

    # Extract and print the reply
    response = resp.content[0].text
    print(response)

# If something fails, catch the error
except Exception as e:
    print("Something went wrong:", e)

#
----

# Lesson 4

## Video 1 — Roles and Conversation History

In [ ]:
# Load environment variables from .env
from dotenv import load_dotenv
load_dotenv()

# Import the Anthropic SDK
import anthropic

# Create the API client
client = anthropic.Anthropic()

In [ ]:
# Start with an empty conversation
messages = []

In [ ]:
# Add a user message
messages.append({
    "role": "user",
    "content": "Hi, my name is Anas."
})

In [ ]:
# Add an assistant message
messages.append({
    "role": "assistant",
    "content": "Nice to meet you, Anas."
})

In [ ]:
# Add a user message
messages.append({
    "role": "user",
    "content": "I love tennis!"
})

In [ ]:
# Inspect the conversation history
messages

## Video 2 — System Prompt

In [ ]:
# Set the assistant's overall behaviour
system = "You are Claude Assist, a concise and helpful assistant." 

In [ ]:
# Start a simple conversation
messages = [
    {
        "role": "user",
        "content": "Explain what Python is."
    }
]

In [ ]:
# Send the system prompt separately from the messages
resp = client.messages.create(
    model="claude-sonnet-5-5",
    max_tokens=200,
    system=system,
    messages=messages
)

In [ ]:
reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

In [ ]:
# Change the assistant's behaviour
system = "You are Claude Assist. Explain technical topics using very short and simple answers." 

In [ ]:
# Ask the same question again with the new system prompt
resp = client.messages.create(
    model="claude-sonnet-5-5",
    max_tokens=200,
    system=system,
    messages=messages
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

In [ ]:
system = "You are Claude Assist. Always answer in French" 

In [ ]:
# Ask the same question again with the new system prompt
resp = client.messages.create(
    model="claude-sonnet-5-5",
    max_tokens=200,
    system=system,
    messages=messages
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

## Video 3 — Parameters

In [ ]:
# Change the assistant's behaviour
system = "You are Claude Assist. Explain technical topics using very short and simple answers." 

In [ ]:
messages

In [ ]:
# Use a moderate temperature
resp = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=40,
    system=system,
    messages=messages,
    extra_body={"temperature": 0.7},
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

## Video 4 — Conversation Memory and Loop

In [ ]:
# Reset the conversation
system = "You are Claude Assist, a concise and helpful assistant."
messages = []

In [ ]:
# Add the first user message
user = "My favourite sport is tennis."

messages.append({
    "role": "user",
    "content": user
})

In [ ]:
# Send the full conversation history
resp = client.messages.create(
    model="claude-sonnet-5-5",
    max_tokens=200,
    system=system,
    messages=messages
)

In [ ]:
# Extract and print Claude's reply
reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

In [ ]:
# Save Claude's reply in the history
messages.append({
    "role": "assistant",
    "content": reply
})

In [ ]:
messages

In [ ]:
# Add a second user message
user = "What is my favourite sport?"

messages.append({
    "role": "user",
    "content": user
})

In [ ]:
messages

In [ ]:
# Resend the full history
resp = client.messages.create(
    model="claude-sonnet-5-5",
    max_tokens=200,
    system=system,
    messages=messages
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

In [ ]:
# Save the second assistant reply
messages.append({
    "role": "assistant",
    "content": reply
})

In [ ]:
messages

## Video 5 — Tokens, Validation and Recap

In [ ]:
# Inspect the final conversation history
messages

In [ ]:
# Inspect token usage from the last response
resp.usage

In [ ]:
import json
print(json.dumps(resp.usage.model_dump(), indent=2))

#
-----

# Lesson 5

This notebook builds from simple prompt improvements to reusable prompt templates in `prompts.py`.

## Video 1 — Why Prompt Wording Matters

In [ ]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Import the Anthropic SDK
import anthropic

# Create the client
client = anthropic.Anthropic()

# Model used in this lesson
MODEL = "claude-sonnet-5-5"

### vague vs specific

In [ ]:
# A vague prompt leaves many possible directions open
vague_prompt = "Tell me about dogs."

resp = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role": "user", "content": vague_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

In [ ]:
# A more specific prompt narrows the task and output
specific_prompt = """
List 3 dog breeds that are suitable for apartment living.

For each breed:
- Give the breed name
- Give one short reason

Keep the answer concise.
"""

resp = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role": "user", "content": specific_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)

## Video 2 — Four Core Prompting Techniques

### Build a Strong Prompt Step by Step

In [ ]:
# Sample context for the next examples
ticket_text = """
I upgraded my subscription yesterday, but my account still shows the free plan.
I have already logged out and back in twice.
"""

In [ ]:
# 1. Clear instruction
prompt = "Summarise the support ticket in one sentence."

print(prompt)

In [ ]:
# 2. Add a role
prompt = """
You are a support agent.

Summarise the support ticket in one sentence.
"""

print(prompt)

In [ ]:
# 3. Add the context and separate it with tags
prompt = f"""
You are a support agent.

Summarise the support ticket in one sentence.

<ticket>
{ticket_text}
</ticket>
"""

print(prompt)

In [ ]:
# 4. Specify the output format
prompt = f"""
You are a support agent.

Summarise the support ticket in one sentence.

<ticket>
{ticket_text}
</ticket>

Return only the summary.
"""

print(prompt)

In [ ]:
# Send the engineered prompt to Claude
resp = client.messages.create(
    model=MODEL,
    max_tokens=150,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

## Video 3 — Few-Shot Prompting

### Zero-Shot First

In [ ]:
# Input we want to classify
text = "The update fixed the issue and everything works now."

# Zero-shot: instruction only
zero_shot_prompt = f"""
Classify the sentiment as positive or negative.

Text: "{text}"
"""

resp = client.messages.create(
    model=MODEL,
    max_tokens=50,
    messages=[
        {"role": "user", "content": zero_shot_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

### Few shot prompting

In [ ]:
text = "The update didn't fix the issue and I'm not happy"

In [ ]:
# Few-shot: show examples of the pattern we want
few_shot_prompt = f"""
Classify the sentiment.

Text: "Great job!" -> positive
Text: "This is broken." -> negative
Text: "The service was excellent." -> positive

Text: "{text}" ->
"""

resp = client.messages.create(
    model=MODEL,
    max_tokens=50,
    messages=[
        {"role": "user", "content": few_shot_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)

## Video 4 — Reusable Prompt Templates

In [ ]:
# Import the reusable templates from prompts.py

from claude_assist.prompts import summarize_prompt, answer_with_context

In [ ]:
# Build a prompt from the basic template
sample_text = """
The engineering team completed the migration on Friday.
The old database has now been placed in read-only mode.
"""

summary_prompt = summarize_prompt(sample_text)

# Inspect the final prompt generated by the function
print(summary_prompt)

In [ ]:
# Send the generated prompt
resp = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[
        {"role": "user", "content": summary_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)

In [ ]:
# Context and question for the advanced template
context = """
The Premium plan includes 100 GB of storage and priority email support.
The Standard plan includes 25 GB of storage and standard email support.
Only the Premium plan includes access to the analytics dashboard.
"""

question = "Which plan includes the analytics dashboard?"

# Build the prompt from prompts.py
context_prompt = answer_with_context(question, context)

# Inspect the final prompt
print(context_prompt)

In [ ]:
# Send the advanced prompt
resp = client.messages.create(
    model=MODEL,
    max_tokens=150,
    messages=[
        {"role": "user", "content": context_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

In [ ]:
# Test the guardrail in the advanced prompt
missing_question = "How much does the Premium plan cost?"

missing_context_prompt = answer_with_context(
    missing_question,
    context
)

resp = client.messages.create(
    model=MODEL,
    max_tokens=150,
    messages=[
        {"role": "user", "content": missing_context_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

#
---

# Lesson 6

## Video 1 — Context Engineering

In [ ]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Import the Anthropic SDK
import anthropic

# Create the client
client = anthropic.Anthropic()

# Model used in this lesson
MODEL = "claude-sonnet-5-5"

### Relevant context vs noise

In [ ]:
# The task only needs a few relevant facts
question = "Why can the customer not access analytics?"

relevant_context = """
The customer is on the Standard plan.
Only the Premium plan includes access to the analytics dashboard.
"""

In [ ]:
# A focused prompt contains only what the task needs
focused_prompt = f"""
Answer the question using only the context below.

<context>
{relevant_context}
</context>

<question>
{question}
</question>

Return a concise answer.
"""

resp = client.messages.create(
    model=MODEL,
    max_tokens=120,
    messages=[
        {"role": "user", "content": focused_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

In [ ]:
# Extra information can add noise without helping the task
noisy_context = """
The customer joined in 2024.
They use dark mode.
Their team has 14 members.
They opened two support tickets last month.
The customer is on the Standard plan.
Only the Premium plan includes access to the analytics dashboard.
Their office is in London.
They usually log in from Chrome.
"""

noisy_prompt = f"""
Answer the question using only the context below.

<context>
{noisy_context}
</context>

<question>
{question}
</question>

Return a concise answer.
"""

resp = client.messages.create(
    model=MODEL,
    max_tokens=120,
    messages=[
        {"role": "user", "content": noisy_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

## Video 2 — The Context Window and Long Conversations

In [ ]:
# Start with a small conversation history
messages = [
    {"role": "user", "content": "My project is called Claude Assist."},
    {"role": "assistant", "content": "Got it."},
]

print("Messages:", len(messages))
messages

In [ ]:
# Add more turns
messages.extend([
    {"role": "user", "content": "It uses Python."},
    {"role": "assistant", "content": "Understood."},
    {"role": "user", "content": "We are adding structured output."},
    {"role": "assistant", "content": "That makes sense."},
])

print("Messages:", len(messages))
messages

In [ ]:
# Each new API call can resend the entire messages list
# More turns = more context sent again on the next request

print(f"The next request would resend {len(messages)} messages.")

## Video 3 — Requesting Structured JSON

In [ ]:
from claude_assist.prompts import extract_fields_prompt

In [ ]:
# Sample text to extract fields from
text = """
Anas is preparing a course lesson about context engineering and structured output.
"""

# Build the prompt from prompts.py
json_prompt = extract_fields_prompt(text)

# Inspect exactly what will be sent
print(json_prompt)

In [ ]:
# Ask Claude to return only the named JSON fields
resp = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role": "user", "content": json_prompt}
    ]
)

# Claude's reply still arrives as text
raw = "".join(block.text for block in resp.content if block.type == "text")
print(raw)

In [ ]:
print(type(raw))

## Video 4 — Parsing JSON Safely

In [ ]:
raw

In [ ]:
# Python's built-in json library parses JSON text
import json

data = json.loads(raw)

print(data)
print(type(data))

In [ ]:
# Access individual fields like a normal Python dictionary
print("Name:", data["name"])
print("Topic:", data["topic"])

### Handle invalid JSON

In [ ]:
from claude_assist.output import parse_json

In [ ]:
# Valid JSON returns a Python dictionary
valid_raw = '{"name": "Anas", "topic": "AI engineering"}'

parse_json(valid_raw)

In [ ]:
# Invalid JSON returns an error object instead of crashing
invalid_raw = "Here is the result: Anas is working on AI engineering."

parse_json(invalid_raw)

### Use the Full Structured-Output Helper

In [ ]:
# output.py can build the prompt, call Claude and parse the reply
from claude_assist.output import extract_fields

result = extract_fields(
    client=client,
    model=MODEL,
    text="Maya is writing documentation about API security."
)

result

### Keep Conversation History in Bounds

In [ ]:
# Import the history helper
from claude_assist.output import trim_history

In [ ]:
# Build a longer sample history
messages = [
    {"role": "user", "content": f"User message {i}"}
    if i % 2 == 0
    else {"role": "assistant", "content": f"Assistant reply {i}"}
    for i in range(14)
]

print("Before:", len(messages))

In [ ]:
# Keep only the most recent 6 messages
trimmed_messages = trim_history(messages, n=6)

print("After:", len(trimmed_messages))
trimmed_messages

In [ ]:
# In a real conversation loop, trim before sending
messages = trim_history(messages, n=10)

print(f"Sending {len(messages)} recent messages to Claude.")

### Final Validation

In [ ]:
# 1. Valid JSON should become a dictionary
print(parse_json('{"name": "Sam", "topic": "Python"}'))

# 2. Invalid JSON should return an error object
print(parse_json("not valid json"))

# 3. History should keep only the most recent messages
print(trim_history(list(range(20)), n=5))

#
---

# Lesson 7

## Video 1 — Lesson Introduction and Model Family

In [ ]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Import libraries
import anthropic
import time

# Import our model helpers
from claude_assist.models import (
    HAIKU_MODEL,
    SONNET_MODEL,
    OPUS_MODEL,
    FABLE_MODEL,
    EFFORT_LEVELS,
    pick_model,
    read_usage,
)

# Create the client
client = anthropic.Anthropic()

### Claude Code vs API

- Claude Code in VS Code can use an eligible Claude subscription.
- Python SDK calls use the Messages API and API billing.

## Video 2 — Haiku, Sonnet, Opus, and Fable

### Haiku 5.5 — Fast Classification

In [ ]:
# A simple high-volume task
haiku_prompt = '''
Classify each support ticket as:
billing, technical, account, or feature_request.

1. We were charged twice this month.
2. The dashboard is blank for all users.
3. Can we add SSO next quarter?
4. I cannot sign in after resetting my password.

Return only the ticket number and category.
'''

resp = client.messages.create(
    model=HAIKU_MODEL,
    max_tokens=200,
    messages=[
        {"role": "user", "content": haiku_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)

### Sonnet 5.5 — Writing in a Given Tone

In [ ]:
# A writing task that needs good style control
sonnet_prompt = '''
Study this writing style:

"Most AI demos are impressive for 5 minutes.
The real question is what happens on day 50."

"Start with the workflow.
Then decide whether you even need an agent."

Now write 2 short posts about choosing the right AI model.

Keep the same direct, technical tone.
No hashtags.
'''

resp = client.messages.create(
    model=SONNET_MODEL,
    max_tokens=300,
    messages=[
        {"role": "user", "content": sonnet_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)

### Opus 5.5 — Harder Engineering Task

In [ ]:
# A more difficult reasoning task
opus_prompt = '''
A Python asyncio service occasionally processes the same job twice.

Explain:
1. what kind of concurrency problem could cause this,
2. why it happens,
3. one safe way to fix it in a single-process asyncio application.

Keep the answer practical.
'''

resp = client.messages.create(
    model=OPUS_MODEL,
    max_tokens=500,
    messages=[
        {"role": "user", "content": opus_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

### Fable 5.1 — Optional Advanced Example

In [ ]:
# Optional prompt example
fable_prompt = '''
Review a company website as if you were advising the founder.

Focus on:
- positioning
- target customer
- trust signals
- conversion path
- the 5 highest-impact improvements

review this website page: https://www.anasriad.com/ai
'''

resp = client.messages.create(
    model=FABLE_MODEL,
    max_tokens=4000,
    tools=[{
        "type": "web_fetch_20260318",   # web fetch version for Fable 5 / 5.1
        "name": "web_fetch",
        "max_uses": 3,
        "allowed_domains": ["anasriad.com"],
    }],
    messages=[{"role": "user", "content": fable_prompt}],
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)
print(resp.stop_reason, resp.usage)

## Video 3 — Choosing and Routing Models

In [ ]:
# Test the routing function
print("Simple:", pick_model("simple"))
print("Standard:", pick_model("standard"))
print("Hard:", pick_model("hard"))
print("Frontier:", pick_model("frontier"))

In [ ]:
# Example task routing
task = "simple"
model = pick_model(task)

print("Task:", task)
print("Chosen model:", model)

In [ ]:
# See the available effort levels
print(EFFORT_LEVELS)

In [ ]:
# Ask Sonnet to use high effort for a harder task
reasoning_prompt = '''
A product has 1,000 users and a 5% monthly churn rate.

Give 3 practical actions the company should test first to reduce churn,
and briefly explain why each one could help.
'''

resp = client.messages.create(
    model=SONNET_MODEL,
    max_tokens=400,
    output_config={"effort": "low"},
    messages=[
        {"role": "user", "content": reasoning_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)

In [ ]:
# Try the same task with low effort
resp_low = client.messages.create(
    model=OPUS_MODEL,
    max_tokens=400,
    output_config={"effort": "xhigh"},
    messages=[
        {"role": "user", "content": reasoning_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)

## Video 4 — Reading Token Usage

In [ ]:
# Make a request we can inspect
resp = client.messages.create(
    model=SONNET_MODEL,
    max_tokens=150,
    messages=[
        {
            "role": "user",
            "content": "Explain model routing in two short sentences."
        }
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

In [ ]:
# Read the token usage
input_tokens, output_tokens = read_usage(resp)

print("Input tokens:", input_tokens)
print("Output tokens:", output_tokens)

In [ ]:
# Simple live timing
start = time.time()

resp = client.messages.create(
    model=SONNET_MODEL,
    max_tokens=150,
    messages=[
        {
            "role": "user",
            "content": "Why can a smaller model be useful in production?"
        }
    ]
)

end = time.time()

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)
print("Time:", round(end - start, 2), "seconds")

## Video 5 — Comparing Models and Cost

In [ ]:
comparison_prompt = '''
A SaaS company has rising churn.

Customers say:
- onboarding takes too long,
- support replies are slow,
- users do not understand one key feature.

Give 3 practical actions the company should test first.
'''

In [ ]:
# Compare Haiku and Sonnet
for model in [HAIKU_MODEL, SONNET_MODEL]:
    start = time.time()

    resp = client.messages.create(
        model=model,
        max_tokens=250,
        messages=[
            {"role": "user", "content": comparison_prompt}
        ]
    )

    end = time.time()
    input_tokens, output_tokens = read_usage(resp)

    print("\nMODEL:", model)
    print("Time:", round(end - start, 2), "seconds")
    print("Input:", input_tokens)
    print("Output:", output_tokens)
    reply = "".join(block.text for block in resp.content if block.type == "text")
    print("Claude:", reply)

#
----

# Lesson 8

## Video 1 — Requirements and Lesson Introduction

In [ ]:
# A simple example requirement
requirement = "The assistant accepts a user message and returns a Claude response."

print(requirement)

## Video 2 — Package Structure and Imports

In [ ]:
# Import functions from the new package
from claude_assist.prompts import summarize_prompt
from claude_assist.models import pick_model

print(pick_model("simple"))

## Video 3 — Configuration and Single Responsibility

In [ ]:
# Import settings from one place
from claude_assist.config import MODEL, MAX_TOKENS, SYSTEM

print("Model:", MODEL)
print("Max tokens:", MAX_TOKENS)
print("System:", SYSTEM)

## Video 4 — Git and GitHub

Useful Git commands for this lesson:

```bash
git status
git init
git add .
git commit -m "Initial commit"
git branch
git switch main
git pull
git push
git restore file.py
git revert <commit>
```

Git records your project history locally.

GitHub is the online place where you can store and share the repository.

#
---

# Lesson 9

## Video 1 — Conversation State and Design

In [ ]:
# A loose history list can be changed from anywhere
history = []

history.append({
    "role": "user",
    "content": "Hello"
})

print(history)

## Video 2 — Building the Conversation Class

In [ ]:
# Import the class we created for Lesson 9
from dotenv import load_dotenv
from claude_assist import config
from claude_assist.conversation import Conversation

load_dotenv()

convo = Conversation(config.SYSTEM)

print("System:", convo.system)
print("History:", convo.history)

In [ ]:
# Send a real message through the Conversation object
reply = convo.send("My name is Alex. Remember it for this conversation.")

print("Claude:", reply)
print("History:", convo.history)

In [ ]:
# The same object keeps the earlier history
reply = convo.send("What is my name?")

print("Claude:", reply)

## Video 3 — Error Handling by Kind

In [ ]:
# Empty input is a problem the user can fix
print(convo.send(""))

## Video 4 — Retry and Clean Main Loop

In [ ]:
# The class now owns the conversation logic
print("Messages stored:", len(convo.history))

for message in convo.history:
    print(message["role"], "->", message["content"][:80])

In [ ]:
convo = Conversation(config.SYSTEM)

while True:
    user = input("You: ")

    if user.lower() == "quit":
        break

    print("Claude:", convo.send(user))

#
---

# Lesson 10

## Video 2 — Defining Your First Tool

In [ ]:
# Import the tool definition and Python function
from claude_assist.tools import CALCULATOR_TOOL, calculate

print(CALCULATOR_TOOL)

In [ ]:
# Test the Python function directly
print(calculate(18, 27, "multiply"))

## Video 3 — Detecting and Running a Tool

In [ ]:
# Set up the Claude client
from dotenv import load_dotenv
import anthropic

from claude_assist import config

load_dotenv()
client = anthropic.Anthropic()

In [ ]:
# Ask a question that should trigger the calculator tool
messages = [
    {
        "role": "user",
        "content": "What is 137 multiplied by 48? Use the calculator tool."
    }
]

resp = client.messages.create(
    model=config.MODEL,
    max_tokens=config.MAX_TOKENS,
    system=config.SYSTEM,
    tools=[CALCULATOR_TOOL],
    messages=messages
)

print("Stop reason:", resp.stop_reason)
print(resp.content)

In [ ]:
# Find the tool request
for block in resp.content:
    if block.type == "tool_use":
        print("Tool:", block.name)
        print("Inputs:", block.input)
        print("ID:", block.id)

In [ ]:
# Run the requested calculator function
for block in resp.content:
    if block.type == "tool_use" and block.name == "calculate":
        args = block.input

        result = calculate(
            args["a"],
            args["b"],
            args["operation"]
        )

        tool_block = block

        print("Result:", result)

## Video 4 — Completing the Full Tool-Use Cycle

In [ ]:
# Keep Claude's tool request in the message history
messages.append({
    "role": "assistant",
    "content": resp.content
})

# Add the calculator result as a tool_result
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_block.id,
        "content": str(result)
    }]
})

In [ ]:
# Call Claude again so it can use the tool result
final = client.messages.create(
    model=config.MODEL,
    max_tokens=config.MAX_TOKENS,
    system=config.SYSTEM,
    tools=[CALCULATOR_TOOL],
    messages=messages
)

reply = "".join(block.text for block in final.content if block.type == "text")
print(reply)

The full cycle is now complete:

1. User asks a question
2. Claude requests the calculator
3. Python runs the calculator
4. We return the `tool_result`
5. Claude gives the final answer

#
---

# Lesson 11

## Video 1 — From One Tool to a Toolbox

In [ ]:
# Import the three tools and their definitions
from claude_assist.tools import (
    calculate,
    get_time,
    count_words,
    TOOL_DEFINITIONS
)

print("Registered tools:")
for tool in TOOL_DEFINITIONS:
    print("-", tool["name"])

In [ ]:
# Test each Python function directly
print("Calculate:", calculate(27, 18, "multiply"))
print("Time:", get_time())
print("Words:", count_words("Claude can now use several tools"))

## Video 2 — Building the Dispatcher

In [ ]:
# Import the dispatcher
from claude_assist.tools import dispatch

print(dispatch(
    "calculate",
    {"a": 12, "b": 7, "operation": "multiply"}
))

In [ ]:
# The same dispatcher can route to a different tool
print(dispatch(
    "count_words",
    {"text": "Claude can choose the right tool for each task"}
))

In [ ]:
# An unknown tool returns a clear error instead of crashing
print(dispatch(
    "missing_tool",
    {}
))

`**args` passes the values from Claude's input dictionary into the matching Python function.

## Video 3 — Handling Multiple Tool Requests

In [ ]:
# Set up the Claude client
from dotenv import load_dotenv
import anthropic

from claude_assist import config

load_dotenv()
client = anthropic.Anthropic()

In [ ]:
# Ask one question that needs two different tools
messages = [{
    "role": "user",
    "content": (
        'What is 27 multiplied by 18, and how many words are in '
        '"Claude can now use several tools"? Use the tools.'
    )
}]

resp = client.messages.create(
    model=config.MODEL,
    max_tokens=config.MAX_TOKENS,
    system=config.SYSTEM,
    tools=TOOL_DEFINITIONS,
    messages=messages
)

print("Stop reason:", resp.stop_reason)

for block in resp.content:
    print(block)

In [ ]:
# Run every tool Claude requested and collect the results
results = []

for block in resp.content:
    if block.type == "tool_use":
        result = dispatch(
            block.name,
            block.input
        )

        results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": str(result)
        })

        print(block.name, "->", result)

In [ ]:
# Keep Claude's tool requests and return all results together
messages.append({
    "role": "assistant",
    "content": resp.content
})

messages.append({
    "role": "user",
    "content": results
})

final = client.messages.create(
    model=config.MODEL,
    max_tokens=config.MAX_TOKENS,
    system=config.SYSTEM,
    tools=TOOL_DEFINITIONS,
    messages=messages
)

print(final.content[0].text)

## Video 4 — Shaping Tool Choice and Validation

In [ ]:
# Valid input works normally
print(calculate(10, 3, "divide"))

In [ ]:
# Valid input works normally
print(calculate(10, 5, "divide"))

# Invalid inputs return clear errors
print(calculate("ten", 5, "add"))
print(calculate(10, 5, "power"))
print(count_words(123))

The goal is simple: bad inputs should fail clearly, not crash the application.

## Video 5 — Lab Validation and Recap

In [ ]:
# Test the full Conversation class with a multi-tool prompt
from claude_assist.conversation import Conversation

convo = Conversation(config.SYSTEM)

reply = convo.send(
    'What is 15 multiplied by 8, and how many words are in '
    '"Tools make Claude more useful"?'
)

print("Claude:", reply)

In [ ]:
# Check that the conversation contains the complete exchange
print("Messages stored:", len(convo.history))

for message in convo.history:
    print(message["role"], "->", message["content"])

#
----

# Lesson 12

## Video 2 — MCP Servers and Clients

In [ ]:
# Confirm the current MCP SDK is installed
import mcp

print("MCP SDK imported successfully")

## Video 3 — Building a Minimal MCP Server

In [ ]:
# Confirm the current MCP SDK is installed
import mcp

print("MCP SDK imported successfully")

In [ ]:
# Import the server object we created
from mcp_server import mcp, NOTES

print("Server:", mcp.name)
print("Current notes:", NOTES)

In [ ]:
# Connect to the server directly in memory for a simple notebook test
from mcp import Client

async with Client(mcp) as client:
    tools = await client.list_tools()

    for tool in tools.tools:
        print(tool.name)

In [ ]:
# Call save_note through MCP
async with Client(mcp) as client:
    result = await client.call_tool(
        "save_note",
        {"text": "Prepare Lesson 13. including labs. Very exciting!"}
    )

    print(result)

In [ ]:
# Call get_notes through MCP
async with Client(mcp) as client:
    result = await client.call_tool(
        "get_notes",
        {}
    )

    print(result)

## Video 4 — Real MCP Examples: Notion and Google Sheets

## MCP Notion

In [5]:
# Generic remote MCP pattern after authentication is configured

# pip install fastmcp
from fastmcp import Client


NOTION_MCP_URL = "https://mcp.notion.com/mcp"

# After OAuth is configured for the connection:
#
async with Client(NOTION_MCP_URL, auth="oauth") as client:
    tools = await client.list_tools()
    print([t.name for t in tools])

/Users/riadanas/Desktop/claude-assist/.venv/lib/python3.14/site-packages/fastmcp/client/auth/oauth.py:269: UserWarning: Using in-memory token storage -- tokens will be lost when the client restarts. For persistent storage across multiple MCP servers, provide an encrypted AsyncKeyValue backend. See https://gofastmcp.com/clients/auth/oauth#token-storage for details.
  self._bind(mcp_url)


[10/25/26 08:26:11] INFO     OAuth authorization URL:                                                  ]8;id=12507129;file:///Users/riadanas/Desktop/claude-assist/.venv/lib/python3.14/site-packages/fastmcp/client/auth/oauth.py\oauth.py]8;;\:]8;id=12507130;file:///Users/riadanas/Desktop/claude-assist/.venv/lib/python3.14/site-packages/fastmcp/client/auth/oauth.py#408\408]8;;\
                             https://mcp.notion.com/authorize?response_type=code&client_id=1K7yJw1Wr7l             
                             VfaX3&redirect_uri=http%3A%2F%2Flocalhost%3A52378%2Fcallback&state=7iii-T             
                             7iJSRRSBHmD3Qji7FGaYe_7FyJPgGYfHyV5Kg&code_challenge=RCPYfXdWGEGGwGWGIgM3             
                             cKfaSvXVi15PLjdY6gk43BI&code_challenge_method=S256&resource=https%3A%2F%2             
                             Fmcp.notion.com%2Fmcp&scope=default                                                   

                    INFO     🎧 OAuth callback server started on http://localhost:52378                ]8;id=12507135;file:///Users/riadanas/Desktop/claude-assist/.venv/lib/python3.14/site-packages/fastmcp/client/auth/oauth.py\oauth.py]8;;\:]8;id=12507136;file:///Users/riadanas/Desktop/claude-assist/.venv/lib/python3.14/site-packages/fastmcp/client/auth/oauth.py#440\440]8;;\

['notion-search', 'notion-memory-search', 'notion-get-tool-access', 'notion-fetch', 'notion-create-attachment', 'notion-create-file-upload', 'notion-download-attachment', 'notion-get-file-download-urls', 'notion-create-pages', 'notion-update-page', 'notion-delete-pages', 'notion-delete-databases', 'notion-restore-pages', 'notion-convert-page-to-skill', 'notion-upload-skill', 'notion-download-skill', 'notion-get-enabled-skills', 'notion-update-enabled-skills', 'notion-move-pages', 'notion-duplicate-page', 'notion-create-database', 'notion-create-folder', 'notion-update-folder', 'notion-update-data-source', 'notion-create-comment', 'notion-get-comments', 'notion-update-comment', 'notion-delete-comment', 'notion-set-discussion-resolved', 'notion-set-comment-reaction', 'notion-get-async-task', 'notion-get-teams', 'notion-get-groups', 'notion-get-users', 'notion-query-data-sources', 'notion-query-multiple-data-sources', 'notion-query-meeting-notes', 'notion-list-private-pages', 'notion-list

In [ ]:
# Install Claude Code CLI
# curl -fsSL https://claude.ai/install.sh | bash

# Then add the MCP connection
# claude mcp add --transport http notion https://mcp.notion.com/mcp

#Summarise the Lesson 12 notebook (assist.ipynb) and create a private Notion page called "Lesson 12: Introducing MCP" with the summary, the server code, and three action items.

## MCP Google Sheets

In [ ]:
# Connecting the Google Sheets MCP server to Claude Code
# Run these in the VS Code terminal, not in this notebook.

# 0. Create a new project in GCP and enable the Sheets API and Sheets MCP API

# Google Sheets remote MCP endpoint
SHEETS_MCP_URL = "https://sheetsmcp.googleapis.com/mcp/v1"

# 1. Add the Google Sheets MCP server (prompts for the client secret)
# claude mcp add --transport http --client-id YOUR_CLIENT_ID --client-secret --callback-port 8080 sheets https://sheetsmcp.googleapis.com/mcp/v1

# 2. Check that the server is registered
# claude mcp list

# 3. Start Claude Code
# claude

# 4. Inside Claude Code, open the MCP menu and authenticate sheets
# /mcp

# 5. Demo prompt
# Using the sheets MCP server, read this spreadsheet: <SPREADSHEET_URL>
# Summarise its content: what the data is, how many rows it has, the main columns, and 3 quick insights.

# To remove the server:
# claude mcp remove sheets

#
----

# Lesson 13

## Video 1 — What an Agent Is

An agent is useful when Claude needs to choose the next action at run time.

A simple way to think about it:

**goal → Claude chooses → app runs action → result returns → repeat or stop**

In [1]:
# Three simple task examples
tasks = {
    "summarise_then_translate": "workflow",
    "classify_support_request": "workflow",
    "research_and_choose_tools": "agent"
}

for task, approach in tasks.items():
    print(task, "->", approach)

summarise_then_translate -> workflow
classify_support_request -> workflow
research_and_choose_tools -> agent


## Video 2 — Workflow or Agent?

Use a workflow when you can write the steps in advance.

Use an agent when the path may need to change based on the task or intermediate results.

In [2]:
# A tiny decision helper for discussion
def choose_approach(steps_known, path_changes):
    if steps_known:
        return "workflow"

    if path_changes:
        return "agent"

    return "workflow"


print(choose_approach(True, False))
print(choose_approach(False, True))

workflow
agent


## Video 3 — Common Workflow Patterns

### Prompt chaining

One step's output becomes the next step's input.

In [3]:
# Simple prompt-chaining idea
article = "Claude can use tools to interact with external systems."

summary = "Claude can use tools with external systems."
translated = "Claude peut utiliser des outils avec des systèmes externes."

print("Summary:", summary)
print("Translated:", translated)

Summary: Claude can use tools with external systems.
Translated: Claude peut utiliser des outils avec des systèmes externes.


### Routing

Your code chooses which path should handle the request.

In [6]:
# Route a request to a known handler
def route_support(category):
    if category == "billing":
        return "Send to the billing flow"

    if category == "technical":
        return "Send to the technical flow"

    return "Send to the general flow"


print(route_support("technical"))

Send to the technical flow


### Parallelization

Independent tasks can be handled separately.

## Video 4 — The Agent Loop and Architecture

In [8]:
MAX_STEPS = 5

# document these
print("Goal: complete a small multi-step task")
print("Tools: local toolbox")
print("Stop: final answer or", MAX_STEPS, "steps")

Goal: complete a small multi-step task
Tools: local toolbox
Stop: final answer or 5 steps


In [9]:
# Import the Lesson 13 scaffold
from claude_assist.agent import run_agent, MAX_STEPS

print("Step limit:", MAX_STEPS)

Step limit: 5


In [10]:
# Concrete example: Claude can answer directly or choose a tool
goal = (
    "Tell me the current local time, then give me one short sentence "
    "explaining why knowing the time could matter before a client call."
)

print(run_agent(goal))

The current local time is 08:22.

Knowing the time helps you join the call on schedule and leaves you enough time to prepare.


## Video 5 — Knowledge Check, Lab and Recap

In [11]:
# A direct-answer example may not need a tool
direct_goal = "In one sentence, explain what an AI agent is."

print(run_agent(direct_goal))

An AI agent is a software system that perceives its environment, makes decisions, and takes actions, often by using tools, to achieve a goal with some degree of autonomy.


In [12]:
# A tool-based example gives Claude a reason to use the toolbox
tool_goal = "What is 27 multiplied by 18? Use the calculator tool."

print(run_agent(tool_goal))

27 × 18 = **486**


#
---

# Lesson 14

## Video 2 — Combining Local and MCP Tools

In [1]:
# Local tools already exist from Lesson 11
from claude_assist.tools import TOOL_DEFINITIONS, TOOLS as LOCAL_TOOLS

print("Local tools:")
for tool in TOOL_DEFINITIONS:
    print("-", tool["name"])

Local tools:
- calculate
- get_time
- count_words


In [2]:
# The MCP notes tool is added to the same toolbox Claude sees
from claude_assist.agent import MCP_NOTE_TOOL, TOOLS

print("Combined tools:")
for tool in TOOLS:
    print("-", tool["name"])

Combined tools:
- calculate
- get_time
- count_words
- save_note


## Video 3 — Guardrails and Stopping

In [3]:
# The loop has a hard cap
from claude_assist.agent import MAX_STEPS

print("Maximum steps:", MAX_STEPS)

Maximum steps: 6


## Video 4 — Tracing and a Real Agent Test

### Meeting Prep Agent

Goal:

> Prepare a concise client-call prep note. Check the current time, keep the final note under 25 words, verify the word count, revise if needed, then save the final note.

Possible path:

1. `get_time`
2. draft a note
3. `count_words`
4. revise if needed
5. `count_words` again
6. `save_note`
7. final answer

The exact path can change because Claude decides what to do next from the results it receives.

In [4]:
meeting_goal = (
    "Prepare a concise client-call prep note. "
    "Check the current local time. "
    "Keep the final note under 25 words. "
    "Use the word-count tool to verify the length. "
    "If it is too long, revise it and check again. "
    "Then save the final note."
)

print(meeting_goal)

Prepare a concise client-call prep note. Check the current local time. Keep the final note under 25 words. Use the word-count tool to verify the length. If it is too long, revise it and check again. Then save the final note.


In [ ]:
# Run the complete agent when your API key and MCP server are ready
from claude_assist.agent import run_agent

# print(run_agent(meeting_goal))
# best to test in python scrip because  asyncio.run() cannot be called from a running event loop
# run this in terminal "python -m claude_assist.agent"